# CareFlow — AI Care Coordination Assistant

**IIT Hyderabad Applied AI Professional Certification · Capstone**

This notebook is one milestone in the eight-milestone CareFlow implementation.
All patient, insurance, appointment and referral examples are synthetic. No real PHI is used.

> **Data boundary:** the official CareFlow Data Toolkit assets are the source of truth.
> Intake records feed M1; mock API tables feed M2/M6; the policy corpus feeds M3/M4;
> the golden evaluation set is finalized in M8 and should be started at M4.

## M7 — Observability, Reliability and Failure Handling

### Goal
Make the CareFlow workflow measurable and resilient.

Track latency, route, tool calls, retrieval count, errors, retries and safety decisions without storing unnecessary patient text.

In [1]:
import time, uuid, json
from dataclasses import dataclass, asdict

@dataclass
class TraceEvent:
    trace_id: str
    step: str
    status: str
    latency_ms: float
    metadata: dict

def traced(step, fn, *args, **kwargs):
    trace_id = kwargs.pop("_trace_id", str(uuid.uuid4()))
    start=time.perf_counter()
    try:
        result=fn(*args,**kwargs)
        status="ok"
        return result, TraceEvent(trace_id,step,status,(time.perf_counter()-start)*1000,{})
    except Exception as e:
        return None, TraceEvent(trace_id,step,"error",(time.perf_counter()-start)*1000,{"error":type(e).__name__})

In [2]:
# Reliability policy: bounded retry + fallback, never infinite retry.
def with_retry(fn, retries=2):
    last=None
    for attempt in range(retries+1):
        try:
            return fn()
        except Exception as e:
            last=e
            if attempt == retries:
                raise
    raise last

events=[]
result, event=traced("health_check", lambda: {"status":"healthy"})
events.append(event)
print(asdict(event))

{'trace_id': '92fa703d-5375-4fb3-b011-45e0d3f2a156', 'step': 'health_check', 'status': 'ok', 'latency_ms': 0.0014159595593810081, 'metadata': {}}


## M7 dashboard signals

| Signal | Why it matters |
|---|---|
| end-to-end latency | user experience |
| tool latency | identify slow backend calls |
| retrieval hit rate | RAG health |
| human-review rate | operational workload |
| clinical-refusal rate | safety |
| error/retry rate | reliability |
| fallback rate | provider resilience |
| citation/source coverage | groundedness |

**Privacy rule:** logs should use IDs and operational metadata rather than storing raw patient messages by default.

In [3]:
# Example SLO-style checks for a notebook demonstration.
sample_metrics = {
    "p95_latency_ms": 850,
    "tool_error_rate": 0.01,
    "human_review_rate": 0.18,
    "clinical_refusal_miss_rate": 0.0,
}
assert sample_metrics["clinical_refusal_miss_rate"] == 0.0
assert sample_metrics["tool_error_rate"] < 0.05
print("Reliability gates passed.")

Reliability gates passed.
